# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Restrict TensorFlow to only allocate 2GB of memory on the first GPU
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
        )
        print("GPU Ready! Physical GPUs:", len(gpus))
    except RuntimeError as e:
        print(e)


2026-11-10 04:01:48.367084: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-10 04:01:48.561479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-10 04:01:48.667386: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-10 04:01:48.687886: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-10 04:01:48.817785: I tensorflow/core/platform/cpu_feature_guar

GPU Ready! Physical GPUs: 1


I0000 00:00:1794283312.112566      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1794283312.198216      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1794283312.198267      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

Num GPUs Available:  1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
tf.__version__

'2.17.0'

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [5]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

In [6]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

### Initialising the CNN

In [7]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [8]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1794283336.692334      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1794283336.692415      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1794283336.692429      65 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1794283336.860974      65 cuda_executor.cc:1001] 

### Step 2 - Pooling

In [9]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [10]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [11]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [12]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Step 5 - Output Layer

In [13]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [14]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [15]:
!pip install scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 769.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 5.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.1.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [16]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1794283377.620385     171 service.cc:146] XLA service 0x7a681c00b7c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794283377.620470     171 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-11-10 04:02:57.665404: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-11-10 04:02:57.894701: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906


  3/250 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.5295 - loss: 0.7265

I0000 00:00:1794283380.746096     171 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 172ms/step - accuracy: 0.5379 - loss: 0.6997 - val_accuracy: 0.6185 - val_loss: 0.6571
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 40s 156ms/step - accuracy: 0.6002 - loss: 0.6679 - val_accuracy: 0.6290 - val_loss: 0.6458
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 153ms/step - accuracy: 0.6495 - loss: 0.6265 - val_accuracy: 0.7020 - val_loss: 0.5964
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 153ms/step - accuracy: 0.6798 - loss: 0.5940 - val_accuracy: 0.7165 - val_loss: 0.5633
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 153ms/step - accuracy: 0.7289 - loss: 0.5497 - val_accuracy: 0.7470 - val_loss: 0.5210
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 153ms/step - accuracy: 0.7439 - loss: 0.5172 - val_accuracy: 0.7075 - val_loss: 0.5458
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 152ms/step - accuracy: 0.7459 - loss: 0.5118 - val_accuracy: 0.7750 - val_loss: 0.4866
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 152ms/step - accuracy: 0.7855 - loss: 0.4569 - val

In [17]:
cnn.save('cat_dog_cnn_model.keras')
print("Model saved successfully!")


Model saved successfully!


## Part 4 - Making a single prediction

In [20]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 871ms/step


In [21]:
print(prediction)

cat


In [23]:
print(result[0][0])

0.0


In [24]:
test_image_2 = image.load_img('single_prediction/cat_or_dog_2.jpg', target_size = (64, 64))
test_image_2 = image.img_to_array(test_image_2)
test_image_2 = np.expand_dims(test_image_2, axis = 0)
result_2 = cnn.predict(test_image_2)
training_set.class_indices
if result_2[0][0] == 1:
  prediction_2 = 'dog'
else:
  prediction_2 = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


In [25]:
print(prediction_2)

dog


In [26]:
print(result_2[0][0])

1.0
